# Exploratory Data Analysis

## About the Dataset

This project uses the Brazilian E-Commerce Public Dataset by Olist.
The dataset contains approximately 100,000 orders made between 2016 and 2018 across multiple marketplaces in Brazil.

It includes information related to orders, customers, products, sellers, payments, logistics and customer reviews,
allowing analysis from multiple perspectives such as sales performance, delivery times and customer satisfaction.

The data is real commercial data, fully anonymized for public use.
    
## Analysis Questions

The main questions this analysis aims to answer are:

1. How have sales evolved over time?
2. Which product categories generate the most revenue?
3. Are there significant differences in sales between states?
4. What is the average delivery time of orders?
5. What proportion of orders are delivered late?
6. Is there a relationship between delivery delays and review scores?
7. Which sellers have the highest sales volume?

In [2]:
#Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
#Import Data
customers=pd.read_csv('../Data/olist_customers_dataset.csv')
geolocation=pd.read_csv('../Data/olist_geolocation_dataset.csv')
order_items=pd.read_csv('../Data/olist_order_items_dataset.csv')
order_payments=pd.read_csv('../Data/olist_order_payments_dataset.csv')
order_reviews=pd.read_csv('../Data/olist_order_reviews_dataset.csv')
orders=pd.read_csv('../Data/olist_orders_dataset.csv')
products=pd.read_csv('../Data/olist_products_dataset.csv')
sellers=pd.read_csv('../Data/olist_sellers_dataset.csv')
translation=pd.read_csv('../Data/product_category_name_translation.csv')

## 1. Product Category Analysis
To understand the business core, I first need to identify which products drive the most revenue. Since the original dataset is in Portuguese, this section involves translating the categories and cleaning the data to ensure an accurate ranking.

**Key Objectives:**
* Harmonize category names using English translations.
* Handle missing values and untranslated categories.
* Identify the Top 10 categories by total sales.


In [65]:
# Check total unique categories in products (excluding NaNs)
print(f"Unique categories in products: {products['product_category_name'].nunique()}")

# Check for products without any assigned category
print(f"Products with missing category name: {products['product_category_name'].isna().sum()}")

# Check total unique translations available
print(f"Unique translations available: {translation['product_category_name_english'].nunique()}")



Unique categories in products: 73
Products with missing category name: 610
Unique translations available: 71


In [61]:
#We merge the product dataset with the English translations. For the missing values in the English column, we fallback to the original Portuguese names. Finally, 
#we handle any remaining nulls as 'Unknown', format the text to Title Case, and retain only the necessary columns.

products_translated = pd.merge(products, translation, on='product_category_name', how='left')

products_translated['product_category_name_english'] = (
    products_translated['product_category_name_english']
    .fillna(products_translated['product_category_name']) 
    .fillna('Unknown')                                   
    .str.replace('_', ' ')                               
    .str.title()                                         
)

products_clean = products_translated[['product_id', 'product_category_name_english']]



We can now use the products_clean table to identify the top-selling categories. To do this, we will merge it with the order_items table, group the data by category, and calculate the total revenue by summing the prices.

In [64]:
# Left merge to bring translated category names into sales data
sales_products = pd.merge(order_items, products_clean, on='product_id', how='left')

# Group by category and sum prices to calculate revenue per category
category_revenue = sales_products.groupby('product_category_name_english')['price'].sum().sort_values(ascending=False)

# Convert to DataFrame and select the top 10 categories
top_10_categories = category_revenue.head(10).reset_index()

# Rename columns for clarity
top_10_categories.columns = ['Category', 'Total Revenue']

print("Top 10 Categories by Revenue:")
print(top_10_categories)

Top 10 Categories by Revenue:
                Category  Total Revenue
0          Health Beauty     1258681.34
1          Watches Gifts     1205005.68
2         Bed Bath Table     1036988.68
3         Sports Leisure      988048.97
4  Computers Accessories      911954.32
5        Furniture Decor      729762.49
6             Cool Stuff      635290.85
7             Housewares      632248.66
8                   Auto      592720.11
9           Garden Tools      485256.46


## Analysis & Insights:

Top Revenue Drivers: The "Health Beauty" and "Watches Gifts" categories are the primary revenue drivers for Olist, both exceeding R$ 1.2M in total sales.

Market Diversity: The Top 10 categories represent a wide range of consumer interests, from essential home goods (Bed Bath Table, Housewares) to discretionary spending (Cool Stuff, Auto).

---
## 2. Sales Evolution Over Time
After identifying the top-performing categories, the next step is to understand the business's growth trajectory. 
In this section, I will analyze how sales have evolved from 2016 to 2018. 

**Key Objectives:**
* Identify the overall sales trend (Growth vs. Stagnation).
* Detect seasonal peaks (e.g., Black Friday impact).
* Determine if the revenue is consistent month-over-month.